# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** OpenCode exited with status 0 without a valid queue-result.json. Last captured output: [2026-10-06T04:27:07.102Z] ! permission requested: external_directory (/home/deliverables/*); auto-rejecting | [2026-10-06T04:27:07.102Z] ✗ cd /home/phyto/phenotyping-colab/runs/p-27b968e6f90cc2d48dfbdf44d668ee61-20261006T040440Z/logs && cp ../deliverables/p-27b968e6f90cc2d48dfbdf44d668ee61_output.ipynb cpu_attempt1_output.ipynb && cp cpu_exec2.log  | [2026-10-06T04:27:07.102Z] Error: The user rejected permission to use this specific tool call. | Required queue-result.json is unavailable or invalid: FileNotFoundError

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: GenoDrawing: An autoencoder framework for image prediction from SNP markers

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-27b968e6f90cc2d48dfbdf44d668ee61)


# GenoDrawing — SNP-to-image prediction (minimal reproduction)

**Paper:** GenoDrawing: An autoencoder framework for image prediction from SNP markers
(bioRxiv 2023.03.06.531351; journal version: Plant Phenomics, doi:10.34133/plantphenomics.0113)

**Author code:** https://github.com/Fedjurrui/GenoDrawing @ commit `f4704eabb7f084d990a6db27b108f17e621bcd8c`
**Weights:** https://huggingface.co/aranzara-lab/genodrawing (MIT)

## Scope of this notebook (partial demonstration)
This notebook runs the authors' own SNP-to-image demo (`GenoDrawing.ipynb`) on 3 example apple
genotypes (12_j001, 2872, 2990): it loads the pretrained encoder/decoder and the two 150-SNP
embedding predictors (targeted tSNPs vs random rSNPs), encodes each genotype's example images to
their mean embedding (ground truth), predicts embeddings from SNPs, decodes them to images, and
composes the 4-column comparison figure. It then re-executes the key metric cells of
`Genodrawing_stats.ipynb` (MAE, randomness thresholds, ECPGR shape-class accuracy/F1,
Wasserstein distances) from the authors' published validation measurements.

**Not reproduced:** autoencoder / predictor training, the full 356-genotype image dataset
(available only "on request" per the paper), and the 100-model SNP-selection experiments.

## 実行スコープ（日本語）
著者自身のデモ `GenoDrawing.ipynb` を 3 遺伝型の例題画像で実行し、tSNP（形質関連 SNP）と
rSNP（ランダム SNP）それぞれから予測画像を生成・比較します。さらに著者公開の検証測定値
（xlsx）を用いて `Genodrawing_stats.ipynb` の主要な評価指標（MAE・ECPGR 形状分類精度・
Wasserstein 距離）を再計算します。学習（オートエンコーダ／予測器の訓練）は対象外です。


## Runtime selection
**Recommended: CPU is sufficient.** The demo runs a small CNN on 27 images of 300×300 pixels plus
tiny MLPs — the authors state the code "has been prepared to run on CPU" and that a GPU merely
improves inference speed. On Colab you may use either `Runtime > Change runtime type > CPU` or
`T4 GPU`; the notebook detects the device and uses the same sample, weights and parameters on
both. Expected wall time: a few minutes on either runtime, dominated by the ~350 MB of weight
downloads.

**ランタイム選択:** 本デモは軽量（300×300 px 画像 27 枚＋小型 MLP）で CPU でも動作します。
GPU（T4）は推論を速めるだけで必須ではありません。どちらのランタイムでも同じ処理を行います。


## 1. Environment setup
The Colab Python 3 kernel already provides torch, torchvision, pandas, numpy, matplotlib,
seaborn, scipy, scikit-learn, Pillow and tqdm. We only ensure `openpyxl` (Excel reader for the
authors' validation measurements) is available.

**説明:** Colab には大半のライブラリが既に揃っているため、xlsx 読み込み用の `openpyxl`
だけを確認・インストールします。

In [ ]:
%pip install -q openpyxl
import torch, torchvision, numpy, pandas, scipy, sklearn, PIL
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Device selection
The authors' notebook selects `cuda` when available and falls back to `cpu`; we keep the same
behavior. Everything downstream uses this `device`.

**説明:** 著者コードと同様に、GPU があれば `cuda`、なければ `cpu` を選択します。

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

## 3. Input and weight acquisition (all downloads happen here, inside Colab)
We pin the exact author repository commit `f4704eab…` and download only the minimal set of
files: the PLINK genotype matrix + SNP lists, the two embedding-predictor state dicts, 3 folders
of example images, the validation-measurement xlsx, and the `GenoDrawingLib` source files.
Encoder/decoder weights (~170 MB each, Git LFS) come from the authors' Hugging Face repo; their
published SHA-256 checksums are verified after download. We deliberately do **not** clone the
whole repository.

**説明:** 著者リポジトリの指定コミットから必要最小限のファイルのみ、Hugging Face
からはエンコーダ／デコーダ重み（SHA-256 検証付き）を Colab 内にダウンロードします。

In [ ]:
import os, json, urllib.request, pathlib

COMMIT = "f4704eabb7f084d990a6db27b108f17e621bcd8c"
REPO = "https://raw.githubusercontent.com/Fedjurrui/GenoDrawing/" + COMMIT

REPO_FILES = [
    "Snp_data/SNPs_dataset.bed",
    "Snp_data/SNPs_dataset.bim",
    "Snp_data/SNPs_dataset.fam",
    "Snp_data/Final_list_snps.csv",
    "Snp_data/list_random.csv",
    "Embedding_predictors/Embedding_targeted/model_data_9_2_2023_15h/embedding_predictor_9_2_2023_15h.h5",
    "Embedding_predictors/Embedding_random/model_data_150_snps_iter_1/150_snps.h5",
    "Measurement_data/genodrawing_validation_data.xlsx",
    "GenoDrawingLib/__init__.py",
    "GenoDrawingLib/models.py",
    "GenoDrawingLib/dataset.py",
]
# Discover the example image file names from the pinned commit's tree (one public API call),
# then fetch only those bytes from pinned raw URLs.
req = urllib.request.Request(
    f"https://api.github.com/repos/Fedjurrui/GenoDrawing/git/trees/{COMMIT}?recursive=1",
    headers={"User-Agent": "colab-reproduction"})
tree = json.load(urllib.request.urlopen(req))["tree"]
example_files = sorted(e["path"] for e in tree
                       if e["path"].startswith("Example_images/") and e["path"].endswith(".jpg"))
print(len(example_files), "example images:", *example_files, sep="\n")

for rel in REPO_FILES + example_files:
    dest = pathlib.Path(rel)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        tmp = dest.with_suffix(dest.suffix + ".part")
        urllib.request.urlretrieve(f"{REPO}/{urllib.parse.quote(rel)}", tmp)
        tmp.rename(dest)
    print(f"{rel}: {dest.stat().st_size} bytes")
print("Repository files downloaded.")

The author example folders contain exactly the files listed above (verified against the pinned
commit tree). Next, the Hugging Face encoder/decoder weights with checksum verification. The
expected SHA-256 values come from the model repository's Git LFS metadata (`encoder_7_2_2023_15h.h5`
= 1af8fa16…, `decoder_7_2_2023_15h.h5` = 7fb4d764…). We use the official `huggingface_hub`
client (preinstalled on Colab), which follows the LFS CDN redirect correctly; plain
`urllib` gets a 401 from the Hub.

**説明:** Hugging Face からエンコーダ／デコーダ重みを取得し、公開されている SHA-256
ハッシュと照合します（約 350 MB、数分）。`huggingface_hub` クライアントを使用します。

In [ ]:
from huggingface_hub import hf_hub_download
import hashlib

HF_FILES = {
    "encoder_7_2_2023_15h.h5":
        "1af8fa163f0d3afce4b00dbe6eff91755116950cd78c1aa542e95874152d7735",
    "decoder_7_2_2023_15h.h5":
        "7fb4d76416aa901e17baf4034e0606e7ace88dc4f40ff053c32c7bce570c79e1",
}
for name, expected in HF_FILES.items():
    local = hf_hub_download(repo_id="aranzara-lab/genodrawing", filename=name,
                            revision="d65531840e6158092f21a5bf1ef22e935a22c7ae",
                            local_dir="AE_model/64_encoders_35_epochs/model_data_7_2_2023_15h")
    dest = pathlib.Path(local)
    h = hashlib.sha256(dest.read_bytes()).hexdigest()
    assert h == expected, f"checksum mismatch for {dest.name}: {h}"
    print(f"{dest.name}: {dest.stat().st_size} bytes, sha256 OK")

## 4. Genotype matrix from the PLINK files
The authors read the PLINK `.bed/.bim/.fam` triple with `pandas_plink` (which returns allele
counts 0/1/2, later divided by 2 to the paper's 0/0.5/1 coding). The pinned legacy
`pandas-plink==2.2.9` conflicts with modern pandas, so this notebook uses a small, documented
replacement reader of the standard PLINK 1 binary format: the file starts with magic
`0x6c 0x1b 0x01`, then each variant's genotypes are bit-packed 2 bits per sample, padded to whole
bytes, in `.fam` sample order. The 2-bit pair values are decoded exactly as pandas-plink 2.2.9's
C reader does (verified against its source): 00→0 (AA), 10→1 (Aa), 11→2 (aa), 01→missing. This is
an AI-assisted
adaptation; the selected 150-SNP values are cross-checked directly against the file below.

**説明:** 著者は `pandas_plink` で PLINK ファイルを読み込みますが、このノートブックでは
同一フォーマットの標準的な NumPy リーダーに置き換えました（0/1/2 → 0/0.5/1 の符号化は
著者コードと同一）。

In [ ]:
import numpy as np, pandas as pd

def read_bim(path):
    cols = ["chrom", "snp", "cM", "pos", "a1", "a2"]
    return pd.read_csv(path, sep=r"\s+", header=None, names=cols, dtype=str)

def read_fam(path):
    cols = ["fid", "iid", "pid", "mid", "sex", "pheno"]
    return pd.read_csv(path, sep=r"\s+", header=None, names=cols, dtype=str)

def read_bed(path, n_samples):
    raw = open(path, "rb").read()
    assert raw[:3] == b"\x6c\x1b\x01", "not a PLINK 1 binary .bed file"
    n_pad = (n_samples + 3) // 4 * 4  # genotypes padded to whole bytes per variant
    bytes_per_variant = n_pad // 4
    body = np.frombuffer(raw[3:], dtype=np.uint8)
    assert len(body) % bytes_per_variant == 0, "truncated .bed"
    body = body.reshape(-1, bytes_per_variant)  # variants x packed bytes
    G = np.empty((body.shape[0], n_pad), dtype=np.float32)
    for k in range(4):
        # pair-bit values verified against pandas-plink 2.2.9's C reader:
        # 00->0 (AA), 10->1 (Aa), 11->2 (aa), 01->missing (NaN)
        two = ((body >> (2 * k)) & 3).astype(np.float32)
        counts = np.full(two.shape, np.nan, dtype=np.float32)
        counts[two == 0] = 0.0
        counts[two == 2] = 1.0
        counts[two == 3] = 2.0
        G[:, k::4] = counts
    return G[:, :n_samples]

bim = read_bim("Snp_data/SNPs_dataset.bim")
fam = read_fam("Snp_data/SNPs_dataset.fam")
G = read_bed("Snp_data/SNPs_dataset.bed", len(fam))
print("variants:", len(bim), "| samples:", len(fam), "| genotype matrix:", G.shape)
print("samples:", fam.iid.tolist())
print("non-missing fraction in file:", float(np.isfinite(G).mean()))

Now we select the authors' 150 targeted (tSNPs) and 150 random (rSNPs) by name, exactly as in
`GenoDrawing.ipynb` cell 1: `snps_df` (targeted) and `snps_df_random`, indexed by lowercased
sample name and holding the paper's 0/0.5/1-coded genotypes.

**説明:** 著者デモと同じく、tSNP/rSNP 各 150 座標を選び、サンプル名（小文字）を索引とする
データフレームに 0/0.5/1 符号化で格納します。

In [ ]:
snp_list = pd.read_csv("Snp_data/Final_list_snps.csv", header=None)[0].tolist()
snp_list_random = pd.read_csv("Snp_data/list_random.csv", header=None)[0].tolist()
print("tSNPs:", len(snp_list), "| rSNPs:", len(snp_list_random))

snp_row = {s: i for i, s in enumerate(bim.snp)}
missing = [s for s in snp_list + snp_list_random if s not in snp_row]
assert not missing, f"SNPs absent from .bim: {missing}"

def build_snp_df(snp_names):
    idx = [snp_row[s] for s in snp_names]
    vals = G[idx].T  # samples x snps
    df = pd.DataFrame(vals, index=[s.lower() for s in fam.iid], columns=snp_names)
    return df

snps_df = build_snp_df(snp_list)
snps_df_random = build_snp_df(snp_list_random)
print(snps_df.shape, snps_df_random.shape)
print("tSNP coding values present:", sorted(np.unique(snps_df.values[~np.isnan(snps_df.values)])))
assert not np.isnan(snps_df.values).any() and not np.isnan(snps_df_random.values).any(), \
    "missing genotype calls in the selected SNPs; the authors' demo requires complete calls"
print(snps_df.iloc[:3, :5])

## 5. Model definitions and pretrained weights
Instead of pasting them, we import the authors' pinned `GenoDrawingLib.models` module (downloaded
in section 3) — the exact network definitions from commit `f4704eab…`. Then we load all four
pretrained state dicts (encoder, decoder, targeted predictor, random predictor) with
`map_location=device` and set them to eval mode, as in the authors' cell 2.

**説明:** 著者の `GenoDrawingLib.models` をピン留したコミットからそのまま読み込み、
4 つの学習済み重みをロードして eval モードにします。

In [ ]:
import sys, torch
sys.path.insert(0, ".")
from GenoDrawingLib.models import Encoder, Decoder, Embedding_predictor

dims = 64
encoder = Encoder(dims=dims)
encoder.load_state_dict(torch.load(
    "AE_model/64_encoders_35_epochs/model_data_7_2_2023_15h/encoder_7_2_2023_15h.h5",
    map_location=device))
decoder = Decoder(dims=dims)
decoder.load_state_dict(torch.load(
    "AE_model/64_encoders_35_epochs/model_data_7_2_2023_15h/decoder_7_2_2023_15h.h5",
    map_location=device))
embedding_predictor_targeted = Embedding_predictor(dims=dims, n_snps=150)
embedding_predictor_targeted.load_state_dict(torch.load(
    "Embedding_predictors/Embedding_targeted/model_data_9_2_2023_15h/embedding_predictor_9_2_2023_15h.h5",
    map_location=device))
embedding_predictor_random = Embedding_predictor(dims=dims, n_snps=150)
embedding_predictor_random.load_state_dict(torch.load(
    "Embedding_predictors/Embedding_random/model_data_150_snps_iter_1/150_snps.h5",
    map_location=device))
encoder.eval(); decoder.eval()
embedding_predictor_targeted.eval(); embedding_predictor_random.eval()
print("All four pretrained state dicts loaded.")

## 6. Input preview
Before the core computation we display the three example genotypes' real apple half-section
images (300×300 RGB, from the pinned repository). These are the actual inputs the reproduction
uses; the genotype name is the sample identifier in the PLINK `.fam`.

**説明:** 推論の前に、再現で実際に使う 3 遺伝型のリンゴ半割画像（著者リポジトリの実入力）
を表示します。

In [ ]:
from PIL import Image

example_dirs = ["Example_images/12_j001", "Example_images/2872", "Example_images/2990"]
first_images = [Image.open(os.path.join(d, sorted(os.listdir(d))[0])) for d in example_dirs]
strip = Image.new("RGB", (300 * 3, 300))
for i, im in enumerate(first_images):
    strip.paste(im.resize((300, 300)) if im.size != (300, 300) else im, (300 * i, 0))
strip

## 7. Ground-truth mean embeddings (authors' `snps_to_embd_dataset`)
We reuse the authors' own `GenoDrawingLib.dataset.snps_to_embd_dataset` unchanged: for each
genotype it encodes all example images with the pretrained encoder, averages the 64-dim
embeddings, and decodes that mean embedding — this decoded image is the genotype's ground-truth
"mean decoded image" used as the predictor's target in the paper (Section 2.3).

**説明:** 著者の `snps_to_embd_dataset` をそのまま使い、各遺伝型の画像をエンコーダで
64 次元埋め込みにして平均し、デコーダで復元した「平均decoded画像」（学習ターゲット相当）を
作成します。

In [ ]:
from GenoDrawingLib.dataset import snps_to_embd_dataset

folders = [os.path.join("Example_images", e) for e in os.listdir("Example_images")]
print(folders)
test_random = snps_to_embd_dataset(folders, snps_df_random, encoder, decoder, device)
test_targeted = snps_to_embd_dataset(folders, snps_df, encoder, decoder, device)

## 8. SNP → image prediction (targeted vs random) and comparison figure
Exactly the authors' cell 4: each genotype's 150 SNPs (0/0.5/1 coding) go through the targeted or
random embedding predictor, and the predicted 64-dim embedding is decoded to a 300×300 image by
the decoder. The composite then shows, per genotype: real example image | decoded mean (ground
truth) | **tSNP prediction** | **rSNP prediction** — the paper's Figure 1 layout.

**説明:** 著者コードと同じ手順で、SNP → 埋め込み予測 → デコードを行い、実画像／平均decoded
画像／tSNP 予測／rSNP 予測の 4 列比較図を作成します。

In [ ]:
from torchvision.transforms.functional import to_pil_image

example_images = [Image.open(os.path.join(d, sorted(os.listdir(d))[0])) for d in example_dirs]
ground_truth_images = [to_pil_image(e[0]) for e in test_random]
images_targeted, images_random = [], []
embedding_predictor_targeted.to(device); embedding_predictor_random.to(device); decoder.to(device)
for e in range(3):
    snp_targeted = torch.reshape(torch.tensor(test_targeted[e][1]), shape=(1, 150))
    snp_random = torch.reshape(torch.tensor(test_random[e][1]), shape=(1, 150))
    prediction_targeted = embedding_predictor_targeted(snp_targeted.to(device)).detach().cpu()
    prediction_random = embedding_predictor_random(snp_random.to(device)).detach().cpu()
    images_targeted.append(to_pil_image(decoder(prediction_targeted.to(device)).detach().cpu()[0]))
    images_random.append(to_pil_image(decoder(prediction_random.to(device)).detach().cpu()[0]))

images_final = []
for i in range(3):
    images_final += [example_images[i], ground_truth_images[i], images_targeted[i], images_random[i]]

composition = Image.new("RGB", (images_final[0].width * 4, images_final[0].height * 3))
for i in range(3):
    for j in range(4):
        composition.paste(images_final[i * 4 + j], (j * 300, i * 300))
from PIL import ImageDraw
draw = ImageDraw.Draw(composition)
for x, label in ((75, "Example images"), (355, "Mean decoded images"),
                 (675, "Targeted version"), (975, "Random version")):
    draw.text((x, 0), label)
os.makedirs("Figures", exist_ok=True)
composition.save("Figures/GenoDrawing_examples.png")
composition

The tSNP (targeted) predictions should resemble the decoded mean images more closely than the
rSNP (random) predictions — the paper's central qualitative claim (Section 3.3). With only 3
genotypes this is a demonstration, not a quantitative reproduction of the paper's statistics.

**説明:** tSNP 予測は rSNP 予測より平均decoded画像に近いはずです（論文 §3.3 の定性結果）。
3 遺伝型のデモであり、統計的な再現ではありません。

## 9. Quantitative evaluation from the authors' validation measurements
`Measurement_data/genodrawing_validation_data.xlsx` holds the authors' published FSI (Fruit Shape
Index External) and SR (Shoulder Ratio) values for 36 validation genotypes, measured on real
images (GT/`_Real_Mean`), tSNP predictions (`_targeted`) and rSNP predictions (`_random`). We
first preview it.

**説明:** 著者が公開している 36 遺伝型分の FSI・SR 検証測定値（xlsx）を読み込み、内容を
確認します。

In [ ]:
df = pd.read_excel("Measurement_data/genodrawing_validation_data.xlsx", index_col=0)
print(df.shape)
df.head()

### 9.1 MAE of FSI/SR, targeted vs random
The authors' cell: MAE against ground truth. Paper values (Section 3.3): FSI MAE 0.0386 (tSNP)
vs 0.0651 (rSNP); SR MAE 0.0317 vs 0.0493.

**説明:** 論文 §3.3 の MAE（FSI 0.0386/0.0651、SR 0.0317/0.0493）を同じコードで再計算します。

In [ ]:
from sklearn.metrics import mean_absolute_error
print("FSIE MAE targeted vs GT: ", mean_absolute_error(df["FSI_targeted"], df["FSI_GT"]))
print("SR MAE targeted vs GT:   ", mean_absolute_error(df["SR_targeted"], df["SR_GT"]))
print("FSIE MAE random vs GT:   ", mean_absolute_error(df["FSI_random"], df["FSI_GT"]))
print("SR MAE random vs GT:     ", mean_absolute_error(df["SR_random"], df["SR_GT"]))

### 9.2 Randomness thresholds
10,000 simulated MAE values against samples drawn from the validation distribution (the authors'
cell 3, seed i for iteration i). Paper: FSI threshold 0.075, SR threshold 0.0529 — both above the
models' errors, so predictions are not random sampling artifacts.

**説明:** 検証分布からの正規乱数サンプリングによる MAE を 10,000 回シミュレートし、
「ランダム性」の閾値（論文: FSI 0.075 / SR 0.0529）を再計算します。

In [ ]:
fsie_mape, sr_mape = [], []
fsies_val = df["FSI_GT"].values
shoulder_ratio_val = df["SR_GT"].values
for i in range(10000):
    np.random.seed(i)
    fsie_mape.append(mean_absolute_error(fsies_val,
        np.random.normal(loc=np.mean(fsies_val), scale=np.std(fsies_val), size=36)))
    sr_mape.append(mean_absolute_error(shoulder_ratio_val,
        np.random.normal(loc=np.mean(shoulder_ratio_val), scale=np.std(shoulder_ratio_val), size=36)))
print("FSI randomness threshold (mean of simulated MAE):", np.mean(fsie_mape), "+/-", np.std(fsie_mape))
print("SR randomness threshold (mean of simulated MAE):  ", np.mean(sr_mape), "+/-", np.std(sr_mape))

### 9.3 ECPGR shape-class accuracy and F1
The authors' `categorize_ecpgr` maps FSI into the five ECPGR shape categories; accuracy and
weighted F1 follow. Paper (Section 3.3): tSNP accuracy 0.61 / F1 0.59; rSNP 0.39 / 0.34.

**説明:** 著者の分類関数で FSI を 5 形状クラスに分け、精度と重み付き F1 を計算します
（論文値: tSNP 0.61/0.59、rSNP 0.39/0.34）。

In [ ]:
def categorize_ecpgr(FSIE):
    if FSIE >= 1.05:
        return "Oval"
    elif FSIE >= 0.95:
        return "Oblong"
    elif FSIE >= 0.85:
        return "Globose"
    elif FSIE >= 0.75:
        return "Flat-Globose"
    else:
        return "Flat"

df_fsi = df.loc[:, ["FSI_targeted", "FSI_random", "FSI_GT"]]
Cat_targeted = df_fsi["FSI_targeted"].apply(categorize_ecpgr)
Cat_random = df_fsi["FSI_random"].apply(categorize_ecpgr)
Cat_GT = df_fsi["FSI_GT"].apply(categorize_ecpgr)
from sklearn.metrics import accuracy_score, f1_score
print("Accuracy of targeted: ", accuracy_score(Cat_GT, Cat_targeted))
print("F1 score of targeted: ", f1_score(Cat_GT, Cat_targeted, average="weighted"))
print("Accuracy of random:   ", accuracy_score(Cat_GT, Cat_random))
print("F1 score of random:   ", f1_score(Cat_GT, Cat_random, average="weighted"))

### 9.4 Confusion matrices (rSNP, then tSNP)
The authors' heatmap cells; the rSNP predictions collapse mostly into the "Flat" category
(Section 3.3).

**説明:** 著者セルと同じ混同行列（ヒートマップ）を rSNP／tSNP の順に描画します。

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
fig = plt.figure(figsize=(8, 8))
ax1 = fig.add_subplot(111)
sns.heatmap(pd.crosstab(Cat_GT, Cat_random), annot=True, ax=ax1, cmap="Blues",
            cbar=False, annot_kws={"size": 18})
ax1.set_xlabel("Predictions", fontsize=16); ax1.set_ylabel("Ground Truth", fontsize=16)
ax1.set_title("Genodrawing random", fontsize=16)
plt.savefig("Figures/confussion_matrix_rand.png"); plt.show()

The same heatmap for the targeted (tSNP) predictions, which should spread across the true
shape categories rather than collapsing into one class.

**説明:** 同じヒートマップを tSNP 予測について描画します（1 クラスへの偏りがないか確認）。

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax1 = fig.add_subplot(111)
sns.heatmap(pd.crosstab(Cat_GT, Cat_targeted), annot=True, ax=ax1, cmap="Blues",
            cbar=False, annot_kws={"size": 18})
ax1.set_xlabel("Predictions", fontsize=16); ax1.set_ylabel("Ground Truth", fontsize=16)
ax1.set_title("Genodrawing targeted", fontsize=16)
plt.savefig("Figures/confussion_matrix_targeted.png"); plt.show()

### 9.5 Wasserstein distances and FSI distribution
Distributional comparison of tSNP vs rSNP predictions against the decoded-mean and genotype-mean
references (the authors' cells 10–12).

**説明:** Wasserstein 距離と FSI 分布（KDE）を著者セルと同じ形式で再計算・描画します。

In [ ]:
from scipy.stats import wasserstein_distance
df_fsi_named = pd.DataFrame({"FSI targeted": df["FSI_targeted"], "FSI random": df["FSI_random"],
                             "FSI decoded mean": df["FSI_GT"], "FSI genotype mean": df["FSI_Real_Mean"]})
df_sr_named = pd.DataFrame({"SR targeted": df["SR_targeted"], "SR random": df["SR_random"],
                            "SR decoded mean": df["SR_GT"], "SR genotype mean": df["SR_Real_Mean"]})
for col in df_fsi_named.columns:
    print("W-distance FSI targeted vs", col, ":",
          wasserstein_distance(df_fsi_named["FSI targeted"], df_fsi_named[col]))
for col in df_sr_named.columns:
    print("W-distance SR targeted vs", col, ":",
          wasserstein_distance(df_sr_named["SR targeted"], df_sr_named[col]))

The FSI distribution plot (KDE) mirrors the authors' `Figures/fsi_distribution.png`: tSNP
predictions should track the decoded-mean and genotype-mean references more closely than rSNPs.

**説明:** FSI 分布の KDE 図を描画します（著者の Figure と同じ形式）。

In [ ]:
g = sns.displot(df_fsi_named, kind="kde", height=8, aspect=0.6, fill=True)
g.set_xlabels("Fruit Shape Index (FSI)")
g.fig.suptitle("Distribution of FSI values")
g.fig.tight_layout(); g.fig.set_dpi(300)
sns.move_legend(g, "upper left", bbox_to_anchor=(0.62, 0.91), frameon=True)
g.fig.savefig("Figures/fsi_distribution.png")
plt.show(); plt.close()

### 9.6 Results table and CSV export
A compact summary of the re-computed metrics (this run's output, in the notebook's runtime).

**説明:** 再計算した指標を表にまとめ、CSV として保存します。

In [ ]:
summary = pd.DataFrame({
    "metric": ["FSI MAE targeted", "FSI MAE random", "SR MAE targeted", "SR MAE random",
               "FSI class accuracy targeted", "FSI class accuracy random"],
    "value": [mean_absolute_error(df["FSI_targeted"], df["FSI_GT"]),
              mean_absolute_error(df["FSI_random"], df["FSI_GT"]),
              mean_absolute_error(df["SR_targeted"], df["SR_GT"]),
              mean_absolute_error(df["SR_random"], df["SR_GT"]),
              accuracy_score(Cat_GT, Cat_targeted),
              accuracy_score(Cat_GT, Cat_random)],
    "paper_reported": [0.0386, 0.0651, 0.0317, 0.0493, 0.61, 0.39],
})
display(summary)
summary.to_csv("genodrawing_metric_summary.csv", index=False)
print("saved genodrawing_metric_summary.csv | device used:", device.type)

## 10. Interpretation and validation record
- The composite figure (Section 8) and the re-computed metrics (Section 9) reproduce the
  authors' published demo and statistics from their own pinned code, weights and measurement
  file. Compare the printed MAE/accuracy values with the `paper_reported` column.
- **AI-assisted adaptations (documented):** (1) the PLINK `.bed` reader is a small numpy
  replacement for the pinned legacy `pandas-plink 2.2.9` (same 0/1/2 → 0/0.5/1 coding); (2)
  metric cells are copied verbatim from `Genodrawing_stats.ipynb`; (3) the composite figure
  labels are drawn without the Windows-only `arial.ttf` (fallback branch of the authors' try/except).
- **Not covered by this demo:** training, the full image dataset (available only on request),
  the 100-model SNP-count experiments, and the (partly unpublished) GWAS provenance of the tSNP set.
- Provenance: repo Fedjurrui/GenoDrawing @ f4704eabb7f084d990a6db27b108f17e621bcd8c;
  HF aranzara-lab/genodrawing @ d65531840e6158092f21a5bf1ef22e935a22c7ae (MIT). Weights'
  SHA-256 verified at download. Genotype data: apple RefPop subset shipped in the repo.

**解釈と検証記録:** 本デモは著者 pinned コード・重み・測定値から著者デモと主要統計を再現
します。AI による適応（PLINK リーダー置換、ラベル描画のフォールバック）は上記の通り明示
済みです。学習や全データセット解析は対象外です。